# Preprocessing a subset of the realistic Dataset
Especially labeling and checking which trajectories should be used will be done here

In [1]:
import numpy as np
import pandas as pd
import sys
import shutil
import os

In [ ]:
def angle_to_axis(vx, vy, vz, zhat):
    axis = np.array((vx, vy, vz))
    axis_norm = axis / np.linalg.norm(axis)
    reference_norm = np.array(zhat) / np.linalg.norm(np.array(zhat))
    cos_angle = np.dot(axis_norm, reference_norm)
    cos_theta = np.clip(cos_angle, -1.0, 1.0)
    angle_degrees = np.degrees(np.arccos(abs(cos_theta)))

    return angle_degrees

In [22]:
# load the dataframe
df = pd.read_csv("/data/lkolmar/datasets/realistic/config/simulation.csv")
# df = df[df["finished"] == True]
print(f"Loaded {len(df)} finished simulations")

Loaded 12712 finished simulations


In [23]:
valid_labels = []
threshold = 20  # degrees
count = 0
for i in range(len(df)):
    sample = df.iloc[i]
    vx, vy, vz = sample["rotation_x"], sample["rotation_y"], sample["rotation_z"]
    pos_x = angle_to_axis(vx, vy, vz, (1, 0, 0))  # backspin
    neg_x = angle_to_axis(vx, vy, vz, (-1, 0, 0)) # topspin

    if min(pos_x, neg_x) < threshold:
        valid_labels.append(i)
        if count > 30:
            # break
            pass
        count += 1

print(f"Found {len(valid_labels)} valid labels out of {len(df)}")

Found 768 valid labels out of 12712


In [18]:
print(valid_labels)

[3179, 3180, 3181, 3182, 3189, 3190, 3765, 3766, 3773, 3774, 3775, 3776, 3779, 3780, 3781, 3782, 3783, 3784, 3789, 3790, 3791, 3792, 3793, 3794, 3804, 3805, 3806, 3807, 3821, 3822, 3823, 3824]


In [13]:
print(valid_labels)

[785, 799, 1108, 1109, 1125, 1126, 1127, 1142, 1143, 1144, 1145, 1159, 1160, 1161, 1173, 1174, 1175, 1185, 1477, 1497, 1498, 1499, 1517, 1518, 1519, 1520, 1521, 1537, 1538, 1539, 1540, 1541]


In [14]:
x, y, z = df.iloc[valid_labels[0]][["rotation_x", "rotation_y", "rotation_z"]]
print(f"Example rotation vector: ({x}, {y}, {z})")
print(f"Angle to backspin axis: {angle_to_axis(x, y, z, (1, 0, 1)):.1f} degrees")
print(f"Angle to topspin axis: {angle_to_axis(x, y, z, (-1, 0, 0)):.1f} degrees")

Example rotation vector: (62.75862068965517, -101.3793103448276, 72.41379310344827)
Angle to backspin axis: 14.3 degrees
Angle to topspin axis: 116.7 degrees


In [5]:
backspin = 0
topspin = 0
rpss = []
high = 0
mid = 0
low = 0
for i in valid_labels:
    sample = df.iloc[i]
    vx, vy, vz = sample["rotation_x"], sample["rotation_y"], sample["rotation_z"]
    if vx > 0:
        label = "backspin"
        backspin += 1
    else:
        label = "topspin"
        topspin += 1

    path = "/data/lkolmar/datasets/realistic/" + sample["path"]
    metadata = pd.read_csv(path + "metadata.csv")
    rps = metadata["rotation_omega"].values[0]
    rpss.append(rps)
    print(f"Index {i}: {label}, {rps:.1f}")

    if rps >= 100:
        high += 1
    elif rps >= 70:
        mid += 1
    else:
        low += 1

print(f"Backspin: {backspin}, Topspin: {topspin}, Mean RPS: {np.mean(rpss):.1f} +/- {np.std(rpss):.1f}")
print(f"Max rps: {np.max(rpss):.1f}, Min rps: {np.min(rpss):.1f}")
print(f"High (>100 rps): {high}, Mid (70-100 rps): {mid}, Low (<70 rps): {low}")

Index 785: backspin, 139.5
Index 799: backspin, 139.5
Index 1108: backspin, 130.5
Index 1109: backspin, 136.8
Index 1125: backspin, 123.2
Index 1126: backspin, 128.4
Index 1127: backspin, 134.0
Index 1142: backspin, 123.2
Index 1143: backspin, 127.6
Index 1144: backspin, 132.6
Index 1145: backspin, 138.2
Index 1159: backspin, 128.4
Index 1160: backspin, 132.6
Index 1161: backspin, 137.5
Index 1173: backspin, 130.5
Index 1174: backspin, 134.0
Index 1175: backspin, 138.2
Index 1185: backspin, 136.8
Index 1477: backspin, 132.6
Index 1497: backspin, 120.9
Index 1498: backspin, 127.6
Index 1499: backspin, 134.7
Index 1517: backspin, 112.1
Index 1518: backspin, 117.8
Index 1519: backspin, 123.9
Index 1520: backspin, 130.5
Index 1521: backspin, 137.5
Index 1537: backspin, 111.2
Index 1538: backspin, 116.2
Index 1539: backspin, 121.7
Index 1540: backspin, 127.6
Index 1541: backspin, 134.0
Index 1556: backspin, 112.1
Index 1557: backspin, 116.2
Index 1558: backspin, 120.9
Index 1559: backspin, 

FileNotFoundError: [Errno 2] No such file or directory: '/data/lkolmar/datasets/realistic/data/05056/05056_metadata.csv'

In [10]:
label_names = {
    "topspin_slow": 0,
    "topspin_mid": 1,
    "topspin_fast": 2,
    "backspin_slow": 3,
    "backspin_mid": 4,
    "backspin_fast": 5,
}

# Select valid balls and copy to new subset dataset

In [11]:
new_index = 0
labels = []
count = 0
for i in valid_labels:
    sample = df.iloc[i]
    vx, vy, vz = sample["rotation_x"], sample["rotation_y"], sample["rotation_z"]
    path = "/data/lkolmar/datasets/realistic/" + sample["path"]
    metadata = pd.read_csv(path + "metadata.csv")
    rps = metadata["rotation_omega"].values[0]

    if vx > 0:
        label = "backspin"
        if rps >= 100:
            labels.append(label_names["backspin_fast"])
        elif rps >= 70:
            labels.append(label_names["backspin_mid"])
        else:
            labels.append(label_names["backspin_slow"])

    else:
        label = "topspin"
        if rps >= 100:
            labels.append(label_names["topspin_fast"])
        elif rps >= 70:
            labels.append(label_names["topspin_mid"])
        else:
            labels.append(label_names["topspin_slow"])

    index_str = str(new_index).zfill(5)
    os.makedirs(f"/data/lkolmar/datasets/subset/preprocessed/{index_str}", exist_ok=True)
    new_path = f"/data/lkolmar/datasets/subset/preprocessed/{index_str}/{index_str}_roi.hdf5"
    # print(path[-6:-1])
    shutil.copyfile(f"/data/lkolmar/datasets/realistic/preprocessed/{path[-6:-1]}/{path[-6:-1]}_roi.hdf5", new_path)
    new_index += 1
    # print(f"Copied {new_path} with label {label} and {rps:.1f} rps")
    # if new_index >= 10:
    # break
    if count > 30:
        break
    count += 1

df_labels = pd.DataFrame({'index': range(len(labels)), 'label': labels})
print(df_labels)

FileNotFoundError: [Errno 2] No such file or directory: '/data/lkolmar/datasets/realistic/preprocessed/01108/01108_roi.hdf5'

In [29]:
df_labels.to_csv("/data/lkolmar/datasets/subset/config/labels.csv", index=False)

## Relable already preprocessed to top or back

In [6]:
lable_names = {
    0: "topspin",
    1: "backspin"
}

In [7]:
labels = []
indices = []
df_subset = pd.read_csv("/data/lkolmar/datasets/subset/config/labels_6_classes.csv")
for i in range(len(df_subset)):
    label = df_subset.iloc[i]["label"]
    if label in [0, 1, 2]:
        labels.append(0)
    else:
        labels.append(1)
    indices.append(i)

df_subset["label"] = labels
df_subset["index"] = indices
df_subset.to_csv("/data/lkolmar/datasets/subset/config/labels.csv", index=False)

In [8]:
print(len(df_subset[df_subset["label"] == 0]))
print(len(df_subset[df_subset["label"] == 1]))

83
324
